# HyQE 질문 생성

In [2]:
import torch
import chromadb
client = chromadb.HttpClient(
    host="3.35.104.197",
    port=10090
)

In [ ]:
collection = client.get_collection("multilingual-e5-large")
contexts = collection.get()["documents"]
len(contexts)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_path = "llama-3.2-1b"
output_dir = "output"

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    ## beam search : 이전 sequence 단어들을 고려하여 next token prediction을 수행한다. ( https://hipster4020.tistory.com/194)
)

In [ ]:
new_tokenizer = AutoTokenizer.from_pretrained(output_dir, padding_side = "left") # 배치추론을 위해 padding_side 지정
base_model.resize_token_embeddings(len(new_tokenizer))  #new_tokenizer로 바꿔야하나?
new_model = PeftModel.from_pretrained(base_model, output_dir) # LoRA 가중치를 가져와 기본 모델에 통합

In [6]:
new_tokenizer.model_max_length = 8192

In [7]:
new_model.generation_config.pad_token_id = new_tokenizer.pad_token_id

In [8]:
# GPU로 모델 이동
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
new_model.to(device)
print(device)

cuda


In [9]:
# 2. 추론 함수 정의
import re
ingredient_name_pattern = re.compile("표준명은 (.+)이고;")
brand_name_pattern = re.compile(r"<p>(.+)\\n브랜드")
product_name_pattern = re.compile("제품명:(.+)\n브랜드")

def generate_questions(texts, max_new_tokens=512):
    # 성분에 대한 문서일 때에는 첫번째 질문이 성분 이름으로 시작하도록 강제함.
    prompt_list = []
    
    for text in texts:
        num_questions = 5 if len(text) < 300 else 7 
        prompt =  f"""아래 instruction에 따라 주어진 context에서 질문을 생성하세요.\nContext: {text}\n\nInstruction: 위 context에서 {num_questions}개의 질문을 생성하세요.\n\nQuestions:"""
    
        if "표준명" in text:
            name = ingredient_name_pattern.findall(text)
            if name:
                name = name[0]
                prompt_list.append(prompt + "1. " + name)
            else:
                prompt_list.append(prompt)
        elif "\\n브랜드" in text:
            name = brand_name_pattern.findall(text)
            if name:
                name = name[0]
                prompt_list.append(prompt + "1. " + name)
            else:
                prompt_list.append(prompt)
        elif "제품명:" in text:
            name = product_name_pattern.findall(text)
            if name:
                name = name[0]
                prompt_list.append(prompt + "1. " + name)
            else:
                prompt_list.append(prompt)
        else:
            prompt_list.append(prompt)
        
    inputs = new_tokenizer(prompt_list, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = new_model.generate(
            **inputs,
            temperature=0.15,
            max_new_tokens=max_new_tokens,
            repetition_penalty=1.05,
            top_p=0.92,
            use_cache=True,
            do_sample=True,
        )
    generated_texts = new_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return [generated_text.split("Questions:")[-1] for generated_text in generated_texts]

In [12]:
generate_questions(contexts[:4])

[' 1. AMOREPACIFIC 2012 Q1 Earnings Release에서 주의 사항은 무엇인가요?\n\n2. AMOREPACIFIC 2012 Q1 Earnings Release에서 2012년 1분기 실적 요약은 무엇인가요?\n\n3. AMOREPACIFIC 2012 Q1 Earnings Release에서 주요 브랜드 매출 성장률은 무엇인가요?\n\n4. AMOREPACIFIC 2012 Q1 Earnings Release에서 해외 화장품 부문 성과는 무엇인가요?\n\n5. AMOREPACIFIC 2012 Q1 Earnings Release에서 MC&S (Mass Cosmetics & Sulloc) 매출은 어떻게 변화했나요?\n\n6. AMOREPACIFIC 2012 Q1 Earnings Release에서 AMOREPACIFIC Corp.는 어떤 회계 정책을 적용하고 있나요?',
 ' 1. AMOREPACIFIC는 언제부터 K-IFRS를 적용했나요?\n\n2. AMOREPACIFIC의 매출액은 얼마나 되나요?\n\n3. AMOREPACIFIC의 영업이익은 얼마나 되나요?\n\n4. AMOREPACIFIC의 순이익은 얼마나 되나요?\n\n5. AMOREPACIFIC의 매출액은 어디에서 발생했나요?\n\n6. AMOREPACIFIC의 영업이익은 어디에서 발생했나요?\n\n7. AMOREPACIFIC의 순이익은 어디에서 발생했나요?',
 ' \n\n1. 아모레퍼시픽 2012년 3분기 실적은 무엇인가요?\n\n2. 아모레퍼시픽 2012년 3분기 매출액은 얼마인가요?\n\n3. 아모레퍼시픽 2012년 3분기 영업이익은 얼마인가요?\n\n4. 아모레퍼시픽 2012년 3분기 매출총이익은 얼마인가요?\n\n5. 아모레퍼시픽 2012년 3분기 판매관리비는 얼마인가요?\n\n6. 아모레퍼시픽 2012년 3분기 영업이익은 얼마인가요?\n\n7. 아모레퍼시픽 2012년 3분기 연결당기순이익은 얼마인가요?',
 ' 1. 아모레퍼시픽의 2012년 4분기 실적은 무엇인가요?\n\n

In [13]:
import gc
from tqdm.autonotebook import tqdm

In [14]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# 4. 배치 단위로 질문 생성 (tqdm 추가)

import json, time
from tqdm import tqdm

batch_size = 2  # GPU 메모리에 맞게 조정
results = []
truncation_length = 8000

# 전체 진행률을 표시하기 위해 tqdm 사용
for i in tqdm(range(0, len(contexts), batch_size)):
    batch_contexts = contexts[i:i + batch_size]
    batch_contexts_truncated = [x[:truncation_length] for x in batch_contexts]
    
    # 배치추론 진행
    try:
        questions = generate_questions(batch_contexts_truncated)
        result = [{"context":context, "question":question} for context, question in zip(batch_contexts, questions)]
        results.extend(result)
    except Exception as e:
        print(f"Error processing context: {contexts[:50]}... - {e}")
        results.extend([{"context": context, "questions": "Error"} for context in batch_contexts])
    
    # 메모리 관리
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# 5. 결과 저장
import json
output_path = "generated_questions_cosmetic.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"질문 생성 완료! 결과가 {output_path}에 저장되었습니다.")
print(f"총 {len(results)}개의 데이터 처리됨.")

In [17]:
results[0]

{'context': '==== 청크 1 ====\n\n# 아모레퍼시픽 2012년 1분기\n\n# 아모레퍼시픽 2012 Q1 실적 발표\n# 경영실적_아모레퍼시픽 2012년 1분기\n## 1. 개요\nAMOREPACIFIC 2012 Q1 Earnings Release  \n\n---\n\n## 2. 주의 사항\n- 해당 자료는 참고용으로만 사용해야 하며, 회계 정책 변화 및 시장 불확실성 등으로 인해 실제 결과와 차이가 발생할 수 있음.  \n- AMOREPACIFIC Corp.는 2011년부터 K-IFRS(국제회계기준)을 적용하고 있음.  \n\n---\n\n## 3. 2012년 1분기 실적 요약\n\n철저한 준비로 불황을 극복하고 견고한 유기적 성장 지속  \n- 1분기 매출 7% 성장한 7,415억 원, 영업이익 2% 증가한 1,504억 원  \n- 브랜드 투자 확대, 적극적인 신제품 출시 및 해외 시장 개척으로 분기 최대 실적 달성  \n\n### 매출 및 영업이익 (단위: 억 원)\n\n| 구분 | Q1 2011 (%) | Q1 2012 (%) | 전년 대비(%) |\n|------|------------|------------|-------------|\n| 매출액 | 6,921 (100.0) | 7,415 (100.0) | 7.1 |\n| 화장품 (국내) | 4,983 (72.0) | 5,170 (69.7) | 3.8 |\n| 화장품 (해외) | 778 (11.2) | 979 (13.2) | 25.8 |\n| MC&S | 1,160 (16.8) | 1,265 (17.1) | 9.1 |\n| 영업이익 | 1,480 (21.4) | 1,504 (20.3) | 1.6 |\n| 화장품 (국내) | 1,324 (26.6) | 1,236 (23.9) | -6.7 |\n| 화장품 (해외) | 20 (2.6) | 53 (5.4) | 161.5 |\n| MC&S | 135 (11.6) | 215 (17.0) | 59.1 |\n| 당기순이익 | 1,127 (16.3

In [ ]:
# 제품명 데이터 보완
for i, result in enumerate(tqdm(results)):
    name = product_name_pattern.findall(result["context"])
    # 제품명이 질문에 들어가 있지 않은 경우 다시 생성
    if name and name[0] not in result['question']:
        output = generate_questions([result['context']])
        result['question'] = output[0]

In [ ]:
# 5. 결과 저장
import json
output_path = "generated_questions.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"질문 생성 완료! 결과가 {output_path}에 저장되었습니다.")
print(f"총 {len(results)}개의 데이터 처리됨.")

In [21]:
# 6. 메모리 정리
del new_model
del base_model
torch.cuda.empty_cache()
gc.collect()

2323